In [ ]:


# Cell 1.1: Core Functions
import pandas as pd
import numpy as np

# ============ Base Functions ============
def EMA(series: pd.Series, n: int) -> pd.Series:
    return series.ewm(span=n, adjust=False).mean()

def REF(series: pd.Series, n: int) -> pd.Series:
    return series.shift(n)

def COUNT(cond: pd.Series, n: int) -> pd.Series:
    return cond.rolling(n, min_periods=1).sum()

# ============ Optimized Core Functions ============
def BARSLAST(cond: pd.Series) -> pd.Series:
    """Optimized BARSLAST (vectorized): Returns the number of bars since the last True condition"""
    cond_arr = cond.fillna(False).to_numpy()
    n = len(cond_arr)
    out = np.full(n, np.nan, dtype=float)
    
    # Find all True positions
    true_indices = np.where(cond_arr)[0]
    
    if len(true_indices) == 0:
        return pd.Series(out, index=cond.index)
    
    # Vectorized distance calculation
    for i, idx in enumerate(true_indices):
        out[idx] = 0.0
        if i < len(true_indices) - 1:
            next_idx = true_indices[i + 1]
            out[idx+1:next_idx] = np.arange(1, next_idx - idx)
        else:
            out[idx+1:] = np.arange(1, n - idx)
    
    return pd.Series(out, index=cond.index)

def _var_window_llv(series: pd.Series, win: pd.Series) -> pd.Series:
    """Variable-window lowest value (optimized)"""
    arr = series.to_numpy().astype(float)
    w = win.to_numpy().astype(float)
    n = len(arr)
    out = np.full(n, np.nan, dtype=float)
    
    for i in range(n):
        if np.isnan(arr[i]) or np.isnan(w[i]):
            continue
        k = int(w[i])
        if k <= 0:
            continue
        start = max(0, i - k + 1)
        out[i] = np.nanmin(arr[start:i+1])
    
    return pd.Series(out, index=series.index)

def _var_window_hhv(series: pd.Series, win: pd.Series) -> pd.Series:
    """Variable-window highest value (optimized)"""
    arr = series.to_numpy().astype(float)
    w = win.to_numpy().astype(float)
    n = len(arr)
    out = np.full(n, np.nan, dtype=float)
    
    for i in range(n):
        if np.isnan(arr[i]) or np.isnan(w[i]):
            continue
        k = int(w[i])
        if k <= 0:
            continue
        start = max(0, i - k + 1)
        out[i] = np.nanmax(arr[start:i+1])
    
    return pd.Series(out, index=series.index)

def _ref_var(series: pd.Series, shift: pd.Series) -> pd.Series:
    """Variable-shift reference (optimized)"""
    arr = series.to_numpy().astype(float)
    sh = shift.to_numpy().astype(float)
    n = len(arr)
    out = np.full(n, np.nan, dtype=float)
    
    # Pre-compute valid indices
    valid_mask = ~(np.isnan(arr) | np.isnan(sh))
    valid_indices = np.where(valid_mask)[0]
    
    for i in valid_indices:
        k = int(sh[i])
        j = i - k
        if j >= 0:
            out[i] = arr[j]
    
    return pd.Series(out, index=series.index)

# ============ Main Signal Computation ============
def compute_signals(df: pd.DataFrame, fast=12, slow=26, sig=9):
    """Compute buy and sell signals using MACD divergence"""
    close = df["Close"].astype(float)

    DIFF = EMA(close, fast) - EMA(close, slow)
    DEA  = EMA(DIFF, sig)
    MACD = (DIFF - DEA) * 2.0

    down_cross0 = (REF(MACD, 1) >= 0) & (MACD < 0)
    up_cross0   = (REF(MACD, 1) <= 0) & (MACD > 0)

    N1  = BARSLAST(down_cross0)
    MM1 = BARSLAST(up_cross0)

    winN = N1 + 1
    winM = MM1 + 1

    CC1   = _var_window_llv(close, winN)
    CC2   = _ref_var(CC1, winM)
    CC3   = _ref_var(CC2, winM)

    DIFL1 = _var_window_llv(DIFF, winN)
    DIFL2 = _ref_var(DIFL1, winM)
    DIFL3 = _ref_var(DIFL2, winM)

    CH1   = _var_window_hhv(close, winM)
    CH2   = _ref_var(CH1, winN)
    CH3   = _ref_var(CH2, winN)

    DIFH1 = _var_window_hhv(DIFF, winM)
    DIFH2 = _ref_var(DIFH1, winN)
    DIFH3 = _ref_var(DIFH2, winN)

    # Buy signal (DXDX) — Bullish MACD divergence: price makes new low but MACD does not
    AAA = (CC1 < CC2) & (DIFL1 > DIFL2) & (REF(MACD,1) < 0) & (DIFF < 0)
    BBB = (CC1 < CC3) & (DIFL1 < DIFL2) & (DIFL1 > DIFL3) & (REF(MACD,1) < 0) & (DIFF < 0)
    CCC = (AAA | BBB) & (DIFF < 0)

    JJJ  = REF(CCC,1).fillna(False) & (REF(DIFF,1).abs() >= (DIFF.abs() * 1.01))
    DXDX = (REF(JJJ,1).fillna(False) == False) & JJJ

    # Sell signal (DBJGXC) — Bearish MACD divergence: price makes new high but MACD does not
    ZJDBL = (CH1 > CH2) & (DIFH1 < DIFH2) & (REF(MACD,1) > 0) & (DIFF > 0)
    GXDBL = (CH1 > CH3) & (DIFH1 > DIFH2) & (DIFH1 < DIFH3) & (REF(MACD,1) > 0) & (DIFF > 0)
    DBBL = (ZJDBL | GXDBL) & (DIFF > 0)

    DBJG   = REF(DBBL,1).fillna(False) & (REF(DIFF,1) >= (DIFF * 1.01))
    DBJGXC = (REF(~DBJG,1).fillna(False)) & DBJG

    return DXDX.fillna(False), DBJGXC.fillna(False)

print("✅ Cell 1.1 complete: Core functions loaded")

✅ Cell 1.1 complete: Core functions loaded


In [2]:
# Cell 1.2: Data Retrieval Functions (Polygon.io Version)
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

# =========================
# Configuration
# =========================
POLYGON_API_KEY = "g56XfZ_JVnyh1GclQGEr485YA4hqAvdE"   # <- Enter your key
POLYGON_BASE    = "https://api.polygon.io"
MARKET_TZ       = "America/New_York"
SHOW_TZ         = "America/Los_Angeles"
YF_SLEEP_SECONDS = 0.05

# =========================
# Core Request Function
# =========================
def _polygon_get(url: str, params: dict) -> dict:
    """Polygon GET request with error handling"""
    params["apiKey"] = POLYGON_API_KEY
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def _polygon_aggs(
    ticker: str,
    multiplier: int,
    timespan: str,
    from_date: str,
    to_date: str,
    adjusted: bool = False,
    limit: int = 50000,
) -> pd.DataFrame:
    """
    Fetch Polygon Aggregates with automatic pagination.
    """
    url = f"{POLYGON_BASE}/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from_date}/{to_date}"
    params = {
        "adjusted": str(adjusted).lower(),
        "sort": "asc",
        "limit": limit,
    }

    results = []
    while url:
        data = _polygon_get(url, params)
        results.extend(data.get("results", []))
        next_url = data.get("next_url")
        if next_url:
            url    = next_url
            params = {}
        else:
            break

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)
    df.index = pd.to_datetime(df["t"], unit="ms", utc=True).dt.tz_convert(MARKET_TZ)
    df = df.rename(columns={"o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"})
    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float)
    time.sleep(YF_SLEEP_SECONDS)
    return df

# =========================
# Fetch Intraday (Minute-Level) Data
# =========================
def fetch_intraday(ticker: str,
                   period_days: int = 60,
                   interval_minutes: int = 15) -> pd.DataFrame:
    """Fetch minute-level bar data"""
    to_date   = datetime.now().strftime("%Y-%m-%d")
    from_date = (datetime.now() - timedelta(days=period_days)).strftime("%Y-%m-%d")

    df = _polygon_aggs(
        ticker      = ticker,
        multiplier  = interval_minutes,
        timespan    = "minute",
        from_date   = from_date,
        to_date     = to_date,
    )

    if df.empty:
        raise RuntimeError(f"{ticker}: Intraday data is empty")

    # Keep only regular trading hours 09:30-15:59
    df = df.between_time("09:30", "16:00", inclusive="left")
    return df

# =========================
# Fetch Daily Data
# =========================
def fetch_daily(ticker: str, period_years: int = 2) -> pd.DataFrame:
    """Fetch daily bar data, timestamps aligned to 16:00 ET"""
    to_date   = datetime.now().strftime("%Y-%m-%d")
    from_date = (datetime.now() - timedelta(days=period_years * 365)).strftime("%Y-%m-%d")

    df = _polygon_aggs(
        ticker     = ticker,
        multiplier = 1,
        timespan   = "day",
        from_date  = from_date,
        to_date    = to_date,
    )

    if df.empty:
        raise RuntimeError(f"{ticker}: Daily data is empty")

    # Align timestamps to 16:00 ET (market close)
    dates = df.index.date
    df.index = pd.DatetimeIndex(
        [pd.Timestamp(d).replace(hour=16, minute=0).tz_localize(MARKET_TZ) for d in dates]
    )
    return df

# =========================
# OHLC Resampling
# =========================
def resample_ohlc(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """Resample OHLC data to a specified timeframe"""
    if df is None or df.empty:
        return pd.DataFrame()

    r = df.resample(rule, label="right", closed="left")
    o = r["Open"].first()
    h = r["High"].max()
    l = r["Low"].min()
    c = r["Close"].last()
    v = r["Volume"].sum()
    out = pd.concat([o, h, l, c, v], axis=1)
    out.columns = ["Open", "High", "Low", "Close", "Volume"]
    out = out.dropna()
    return out

# =========================
# Build Multi-Timeframe Data
# =========================
def build_timeframes(intra: pd.DataFrame, daily: pd.DataFrame):
    """Build multi-timeframe OHLC data from intraday and daily bars"""
    weekly = resample_ohlc(daily, "W-FRI")

    timeframes = {
        "30m": resample_ohlc(intra, "30min"),
        "1H":  resample_ohlc(intra, "1h"),
        "2H":  resample_ohlc(intra, "2h"),
        "3H":  resample_ohlc(intra, "3h"),
        "4H":  resample_ohlc(intra, "4h"),
        "D":   daily,
        "W":   weekly,
    }
    return timeframes

print("✅ Cell 1.2 complete: Data retrieval functions loaded (Polygon.io version)")

✅ Cell 1.2 complete: Data retrieval functions loaded (Polygon.io version)


In [3]:
# Cell 1.3: Stock Scanning (Concurrent Version)
import pandas as pd
import time
import os
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# =========================
# Ticker Universe
# =========================
scanning_target = ["QQQ","SPY","SOXX","AAPL","ABBV","ABNB","ABT","ACGL","ACN","ADBE","ADI","ADM","ADP","ADSK","AEE","AEP","AES","AFL","AIG","AIZ","AJG","AKAM","ALB","ALGN","ALL","ALLE","AMAT","AMCR","AMD","AME","AMGN","AMP","AMT","AMZN","ANET","ANSS","AON","AOS","APA","APD","APH","APTV","ARE","ATO","AVB","AVGO","AVY","AWK","AXON","AXP","AZO",
"BA","BAC","BALL","BAX","BBWI","BBY","BDX","BEN","BF-B","BIIB","BK","BKNG","BKR","BLK","BMY","BR","BRK-B","BRO","BSX","BWA","BX","BXP",
"C","CAG","CAH","CARR","CAT","CB","CBOE","CBRE","CCI","CCL","CDNS","CDW","CE","CEG","CF","CFG","CHD","CHRW","CHTR","CI","CINF","CL","CLX","CMA","CMCSA","CME","CMG","CMI","CMS","CNC","CNP","COF","COO","COP","COST","CPB","CPRT","CPT","CRL","CRM","CRWD","CSCO","CSGP","CSX","CTAS","CTLT","CTRA","CTSH","CTVA","CVS","CVX",
"D","DAL","DD","DE","DFS","DG","DGX","DHI","DHR","DIS","DLR","DLTR","DOV","DOW","DPZ","DRI","DTE","DUK","DVA","DVN","DXCM",
"EA","EBAY","ECL","ED","EFX","EIX","EL","ELV","EMN","EMR","ENPH","EOG","EPAM","EQIX","EQR","EQT","ES","ESS","ETN","ETR","EVRG","EW","EXC","EXPD","EXPE","EXR",
"F","FANG","FAST","FCX","FDS","FDX","FE","FFIV","FICO","FIS","FISV","FITB","FLT","FMC","FOXA","FRT","FSLR",
"GD","GE","GEHC","GEN","GENE","GILD","GIS","GL","GLW","GM","GOOGL","GPC","GPN","GRMN","GS","GWW",
"HAL","HAS","HBAN","HCA","HD","HES","HIG","HII","HLT","HOLX","HON","HPE","HPQ","HRL","HSIC","HST","HSY","HUBB","HUM","HWM",
"IBM","ICE","IDXX","IEX","IFF","ILMN","INCY","INTC","INTU","IP","IPG","IQV","IR","IRM","ISRG","IT","ITW","IVZ",
"J","JBHT","JCI","JKHY","JNJ","JPM",
"K","KDP","KEY","KEYS","KHC","KIM","KLAC","KMB","KMI","KMX","KO","KR",
"L","LDOS","LEN","LH","LHX","LIN","LKQ","LLY","LMT","LNT","LOW","LRCX","LULU","LUV","LVS","LW","LYB","LYV",
"MA","MAA","MAR","MAS","MCD","MCHP","MCK","MCO","MDLZ","MDT","MET","META","MGM","MHK","MKC","MKTX","MLM","MMC","MMM","MNST","MO","MOS","MPC","MPWR","MRK","MRNA","MRO","MS","MSCI","MSFT","MSI","MTB","MTD","MU",
"NCLH","NDAQ","NEE","NEM","NFLX","NI","NKE","NOC","NOW","NRG","NSC","NTAP","NTRS","NUE","NVDA","NVR","NWL","NWSA","NXPI",
"O","ODFL","OKE","OMC","ON","ORCL","ORLY","OTIS","OXY",
"PANW","PARA","PAYC","PAYX","PCAR","PCG","PEAK","PEG","PEP","PFE","PFG","PG","PGR","PH","PHM","PKG","PLD","PM","PNC","PNR","PNW","PODD","POOL","PPG","PPL","PRU","PSA","PSX","PTC","PWR","PYPL",
"QCOM",
"RCL","REG","REGN","RF","RHI","RJF","RL","RMD","ROK","ROL","ROP","ROST","RSG","RTX",
"SBUX","SCHW","SEDG","SEE","SHW","SJM","SLB","SMCI","SNA","SNPS","SO","SPG","SPGI","SRE","STE","STT","STX","STZ","SWK","SWKS","SYF","SYK","SYY",
"T","TAP","TDG","TDY","TECH","TEL","TER","TFC","TFX","TGT","TJX","TMO","TMUS","TPR","TRGP","TRMB","TROW","TRV","TSCO","TSLA","TSN","TT","TTWO","TXN","TXT","TYL",
"UAL","UDR","UHS","ULTA","UNH","UNP","UPS","URI","USB",
"V","VFC","VICI","VLO","VMC","VRSK","VRSN","VRTX","VTR","VTRS",
"WAB","WAT","WBA","WBD","WDC","WEC","WELL","WFC","WHR","WM","WMB","WMT","WRB","WY","WYNN","USAR","MP","LAC",
"XEL","XOM","XYL",
"YUM",
"ZBH","ZBRA","ZTS","BJ"
]

N_DAYS = 5
FAST, SLOW, SIG = 12, 26, 9
MAX_WORKERS = 30

# =========================
# Helper Functions
# =========================
def last_n_sessions(idx: pd.DatetimeIndex, n=5):
    dates = pd.Index(idx.tz_convert(MARKET_TZ).date).unique()
    dates = sorted(dates)
    return dates[-n:]

# =========================
# Single Ticker Processing
# =========================
print_lock = threading.Lock()
counter = {"done": 0}

def scan_ticker(ticker: str) -> list:
    """Process a single ticker and return a list of signal rows"""
    local_rows = []
    try:
        intra = fetch_intraday(ticker)
        daily = fetch_daily(ticker)
        tfs   = build_timeframes(intra, daily)

        for tf_name, df_tf in tfs.items():
            if tf_name == "W":
                continue
            if df_tf is None or df_tf.empty or len(df_tf) < 60:
                continue

            bottom, sell_sig = compute_signals(df_tf, FAST, SLOW, SIG)
            days = last_n_sessions(df_tf.index, N_DAYS)

            for d in days:
                start = pd.Timestamp(d).tz_localize(MARKET_TZ)
                end   = start + pd.Timedelta(days=1)
                mask  = (df_tf.index >= start) & (df_tf.index < end)

                if bottom[mask].any():
                    local_rows.append({"Ticker": ticker, "Signal": "BUY", "TF": tf_name, "Date": d.isoformat()})
                if sell_sig[mask].any():
                    local_rows.append({"Ticker": ticker, "Signal": "SELL",  "TF": tf_name, "Date": d.isoformat()})

    except Exception as e:
        local_rows.append({"Ticker": ticker, "Signal": "ERROR", "TF": "-", "Date": str(e)})

    with print_lock:
        counter["done"] += 1
        print(f"[{counter['done']}/{len(scanning_target)}] ✓ {ticker}        ", end="\r")

    return local_rows

# =========================
# Concurrent Scanning
# =========================
rows = []
error_count = 0
counter["done"] = 0
t0 = time.time()

print(f"🚀 Starting scan on {len(scanning_target)} tickers ({MAX_WORKERS} threads)...")
print(f"📅 Scanning last {N_DAYS} trading days")
print(f"📊 Timeframes: Daily / 4H / 3H / 2H / 1H / 30min\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(scan_ticker, t): t for t in scanning_target}
    for future in as_completed(futures):
        result = future.result()
        for r in result:
            if r["Signal"] == "ERROR":
                error_count += 1
        rows.extend(result)

elapsed = round(time.time() - t0, 1)
print(f"\n✅ Scan complete! Elapsed: {elapsed}s")
if error_count > 0:
    print(f"⚠️  {error_count} ticker(s) failed to download")

# =========================
# Build Pivot Table (Wide Format)
# =========================
df = pd.DataFrame(rows)

if df.empty:
    wide = pd.DataFrame(columns=["Ticker", "Date", "Daily", "4H", "3H", "2H", "1H", "30min"])
    print("⚠️  No signals found")
else:
    df = df[df["Signal"].isin(["BUY", "SELL"])].copy()

    tf_rename = {"D": "Daily", "4H": "4H", "3H": "3H", "2H": "2H", "1H": "1H", "30m": "30min"}
    df["TF"] = df["TF"].map(lambda x: tf_rename.get(x, x))

    wide = (
        df.pivot_table(
            index=["Ticker", "Date"],
            columns="TF",
            values="Signal",
            aggfunc=lambda x: "/".join(sorted(set(x)))
        )
        .reset_index()
    )

    tf_cols = ["Daily", "4H", "3H", "2H", "1H", "30min"]
    for c in tf_cols:
        if c not in wide.columns:
            wide[c] = ""

    wide = wide[["Ticker", "Date"] + tf_cols]
    wide.sort_values(["Date", "Ticker"], ascending=[False, True], inplace=True)
    wide.reset_index(drop=True, inplace=True)
    wide = wide.fillna("")

# =========================
# Generate Display Version
# =========================
DISPLAY_MAP = {"BUY": "Buy", "SELL": "Sell"}
wide_display = wide.copy()
for c in ["Daily", "4H", "3H", "2H", "1H", "30min"]:
    if c in wide_display.columns:
        wide_display[c] = wide_display[c].replace(DISPLAY_MAP)

# =========================
# Export CSV
# =========================
folder = "scan_results"
os.makedirs(folder, exist_ok=True)

filename = os.path.join(folder, datetime.now().strftime("%Y-%m-%d") + "_reversal_scan.csv")
wide_display.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"📄 Exported: {filename}")
print(f"📊 Total signal combinations found: {len(wide)}\n")

display(wide_display.head(20))

🚀 Starting scan on 480 tickers (30 threads)...
📅 Scanning last 5 trading days
📊 Timeframes: Daily / 4H / 3H / 2H / 1H / 30min

[480/480] ✓ XYL          
✅ Scan complete! Elapsed: 119.4s
⚠️  16 ticker(s) failed to download
📄 Exported: scan_results/2026-04-01_reversal_scan.csv
📊 Total signal combinations found: 651



TF,Ticker,Date,Daily,4H,3H,2H,1H,30min
0,ACGL,2026-04-01,,,,,,Sell
1,ADI,2026-04-01,,,,,Sell,
2,ADM,2026-04-01,,Sell,Sell,,,
3,ADP,2026-04-01,,,,,Buy,Buy
4,ADSK,2026-04-01,,,,,,Sell
5,AEE,2026-04-01,,,,,,Sell
6,AJG,2026-04-01,,,,Sell,Sell,Sell
7,AMT,2026-04-01,,,,,,Sell
8,AON,2026-04-01,,,,,,Sell
9,APD,2026-04-01,Sell,,,,,


In [4]:
# Cell 2.1: Backtesting Statistics (For Stocks with Buy Signals)
import warnings
import os
import time
import numpy as np
import pandas as pd
import threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings("ignore")

# =========================
# Backtest Configuration
# =========================
BACKTEST_YEARS = 3
HOLD_DAYS      = [3, 5, 8]

# =========================
# Custom Configuration (Optional)
# Leave [] to auto-filter from wide_display for latest trading day Daily/4H buy signals
# =========================
CUSTOM_TICKERS = []              # e.g. ["AAPL", "NVDA"] or leave [] for auto-filter
CUSTOM_TF      = ["Daily","4H"]  # Options: ["Daily"] / ["4H"] / ["Daily", "4H"]

# =========================
# Determine Target Tickers and Timeframes
# =========================
today_str    = datetime.now().strftime("%Y-%m-%d")
last_trading = wide_display["Date"].max()

if CUSTOM_TICKERS:
    target_tickers = CUSTOM_TICKERS
    print(f"📌 Custom mode: {target_tickers}")
else:
    target_tickers = (
        wide_display[
            (wide_display["Date"] == last_trading) &
            (wide_display[["Daily", "4H"]].isin(["Buy", "BUY", "Buy/Sell", "BUY/SELL"])).any(axis=1)
        ]["Ticker"]
        .unique()
        .tolist()
    )
    print(f"📋 Auto mode: Latest trading day ({last_trading}) buy signals")

if not target_tickers:
    print(f"⚠️  No target tickers found, skipping backtest")
else:
    print(f"📊 Target tickers: {target_tickers}")
    print(f"📈 Timeframes: {CUSTOM_TF}")
    print(f"Total: {len(target_tickers)} tickers, starting backtest...\n")

# =========================
# Fetch Backtest Data
# =========================
def fetch_daily_bt(ticker):
    to_date   = datetime.now().strftime("%Y-%m-%d")
    from_date = (datetime.now() - timedelta(days=BACKTEST_YEARS * 365 + 60)).strftime("%Y-%m-%d")
    df = _polygon_aggs(ticker, 1, "day", from_date, to_date)
    if df.empty:
        raise RuntimeError(f"{ticker}: Daily data is empty")
    dates = df.index.date
    df.index = pd.DatetimeIndex(
        [pd.Timestamp(d).replace(hour=16, minute=0).tz_localize(MARKET_TZ) for d in dates]
    )
    return df

def fetch_intraday_bt(ticker):
    to_date   = datetime.now().strftime("%Y-%m-%d")
    from_date = (datetime.now() - timedelta(days=BACKTEST_YEARS * 365 + 60)).strftime("%Y-%m-%d")
    df = _polygon_aggs(ticker, 1, "hour", from_date, to_date)
    if df.empty:
        raise RuntimeError(f"{ticker}: Hourly data is empty")
    df = df.between_time("09:30", "16:00", inclusive="left")
    r   = df.resample("4h", offset="9h30min", label="right", closed="left")
    out = pd.concat([r["Open"].first(), r["High"].max(), r["Low"].min(),
                     r["Close"].last(), r["Volume"].sum()], axis=1)
    out.columns = ["Open", "High", "Low", "Close", "Volume"]
    return out.dropna()

# =========================
# Single Ticker Backtest
# =========================
print_lock = threading.Lock()
bt_counter = {"done": 0}

def backtest_ticker(ticker):
    records = []
    try:
        daily = fetch_daily_bt(ticker)
        intra = fetch_intraday_bt(ticker) if "4H" in CUSTOM_TF else None

        # Daily signals
        if "Daily" in CUSTOM_TF and len(daily) >= 60:
            sig_d = compute_signals(daily, FAST, SLOW, SIG)[0]
            for sig_dt in daily.index[sig_d]:
                records.append({"ticker": ticker, "tf": "Daily", "sig_date": sig_dt, "daily": daily})

        # 4H signals
        if "4H" in CUSTOM_TF and intra is not None and len(intra) >= 60:
            sig_4h = compute_signals(intra, FAST, SLOW, SIG)[0]
            seen_dates = set()
            for bar_dt in intra.index[sig_4h]:
                trade_date = bar_dt.date()
                if trade_date in seen_dates:
                    continue
                seen_dates.add(trade_date)
                sig_dt = pd.Timestamp(trade_date).replace(hour=16, minute=0).tz_localize(MARKET_TZ)
                records.append({"ticker": ticker, "tf": "4H", "sig_date": sig_dt, "daily": daily})

    except Exception as e:
        with print_lock:
            print(f"\n⚠️  {ticker} skipped: {e}")

    with print_lock:
        bt_counter["done"] += 1
        print(f"[{bt_counter['done']}/{len(target_tickers)}] ✓ {ticker}        ", end="\r")

    return records

# =========================
# Calculate Holding Period Returns
# =========================
def calc_returns(records):
    rows = []
    for rec in records:
        ticker, tf, sig_dt, daily = rec["ticker"], rec["tf"], rec["sig_date"], rec["daily"]
        future = daily[daily.index > sig_dt]
        if len(future) < max(HOLD_DAYS):
            continue
        entry_price = future.iloc[0]["Open"]
        if pd.isna(entry_price) or entry_price <= 0:
            continue
        row = {"Ticker": ticker, "TF": tf, "Signal Date": sig_dt.strftime("%Y-%m-%d"), "Entry Price": round(entry_price, 4)}
        for h in HOLD_DAYS:
            if h >= len(future):
                row[f"{h}d Return%"]      = np.nan
                row[f"{h}d Max Gain%"]    = np.nan
                row[f"{h}d Max Drawdown%"] = np.nan
                row[f"{h}d Peak Day"]     = np.nan
            else:
                exit_price  = future.iloc[h]["Close"]
                hold_window = future.iloc[:h+1]
                row[f"{h}d Return%"]      = round((exit_price - entry_price) / entry_price * 100, 2)
                row[f"{h}d Max Gain%"]    = round((hold_window["High"].max() - entry_price) / entry_price * 100, 2)
                row[f"{h}d Max Drawdown%"] = round((hold_window["Low"].min()  - entry_price) / entry_price * 100, 2)
                row[f"{h}d Peak Day"]     = int(hold_window["High"].argmax())
        rows.append(row)
    return pd.DataFrame(rows)

# =========================
# Summary Statistics (Sorted by 3d Win Rate)
# =========================
def summarize(df_trades):
    tables = {}
    for tf in CUSTOM_TF:
        sub_tf = df_trades[df_trades["TF"] == tf]
        if sub_tf.empty:
            continue
        rows = []
        for ticker, sub in sub_tf.groupby("Ticker"):
            row = {"Ticker": ticker, "Samples": len(sub)}
            for h in HOLD_DAYS:
                valid = sub[f"{h}d Return%"].dropna()
                if len(valid) == 0:
                    row[f"{h}d Win Rate"]       = "-"
                    row[f"{h}d Avg Return"]     = "-"
                    row[f"{h}d Max Gain"]       = "-"
                    row[f"{h}d Max Drawdown"]   = "-"
                    row[f"{h}d Avg Peak Day"]   = "-"
                    row[f"{h}d Valid Samples"]   = 0
                else:
                    row[f"{h}d Win Rate"]       = f"{(valid > 0).sum() / len(valid) * 100:.1f}%"
                    row[f"{h}d Avg Return"]     = f"{valid.mean():.2f}%"
                    row[f"{h}d Max Gain"]       = f"{sub[f'{h}d Max Gain%'].dropna().max():.2f}%"
                    row[f"{h}d Max Drawdown"]   = f"{sub[f'{h}d Max Drawdown%'].dropna().min():.2f}%"
                    row[f"{h}d Avg Peak Day"]   = f"{sub[f'{h}d Peak Day'].dropna().mean():.1f}d"
                    row[f"{h}d Valid Samples"]   = len(valid)
            rows.append(row)
        df = pd.DataFrame(rows)
        df["_sort"] = df["3d Win Rate"].replace("-", "0%").str.rstrip("%").astype(float)
        df = df.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)
        tables[tf] = df
    return tables

# =========================
# Main Backtest Flow
# =========================
if target_tickers:
    all_records = []
    bt_counter["done"] = 0
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(backtest_ticker, t): t for t in target_tickers}
        for future in as_completed(futures):
            all_records.extend(future.result())

    elapsed = round(time.time() - t0, 1)
    print(f"\n✅ Backtest complete! Elapsed: {elapsed}s")
    print(f"📈 Found {len(all_records)} historical signal records\n")

    df_trades = calc_returns(all_records)
    df_tables = summarize(df_trades)

    if "Daily" in df_tables:
        print("=" * 60)
        print(f"📊 Daily Buy Signal Backtest Summary ({last_trading})")
        print("=" * 60)
        display(df_tables["Daily"])

    if "4H" in df_tables:
        print("=" * 60)
        print(f"📊 4H Buy Signal Backtest Summary ({last_trading})")
        print("=" * 60)
        display(df_tables["4H"])

    # Export: Both tables in one CSV, separated by blank line
    folder = "scan_results"
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, f"{today_str}_backtest_summary.csv"), "w", encoding="utf-8-sig", newline="") as f:
        if "Daily" in df_tables:
            f.write("Daily Buy Signal Backtest\n")
            df_tables["Daily"].to_csv(f, index=False)
            f.write("\n")
        if "4H" in df_tables:
            f.write("4H Buy Signal Backtest\n")
            df_tables["4H"].to_csv(f, index=False)

    print(f"\n📄 Exported to 'scan_results' folder")

📋 Auto mode: Latest trading day (2026-04-01) buy signals
📊 Target tickers: ['DXCM', 'ETN', 'ISRG', 'KMB', 'MPWR', 'NCLH', 'PTC', 'SWKS']
📈 Timeframes: ['Daily', '4H']
Total: 8 tickers, starting backtest...

[8/8] ✓ NCLH        
✅ Backtest complete! Elapsed: 9.3s
📈 Found 186 historical signal records

📊 Daily Buy Signal Backtest Summary (2026-04-01)


,Ticker,Samples,3d Win Rate,3d Avg Return,3d Max Gain,3d Max Drawdown,3d Avg Peak Day,3d Valid Samples,5d Win Rate,5d Avg Return,5d Max Gain,5d Max Drawdown,5d Avg Peak Day,5d Valid Samples,8d Win Rate,8d Avg Return,8d Max Gain,8d Max Drawdown,8d Avg Peak Day,8d Valid Samples
0,SWKS,8,62.5%,1.55%,10.33%,-9.35%,1.9d,8,25.0%,-0.58%,11.90%,-9.35%,2.4d,8,50.0%,1.21%,13.00%,-9.35%,4.0d,8
1,KMB,12,50.0%,-0.06%,4.64%,-3.33%,1.1d,12,58.3%,-0.17%,4.64%,-3.77%,2.5d,12,66.7%,0.38%,5.21%,-5.52%,4.6d,12
2,MPWR,2,50.0%,-1.58%,6.61%,-10.89%,1.0d,2,0.0%,-4.60%,6.61%,-11.48%,1.0d,2,100.0%,3.09%,6.61%,-11.48%,5.0d,2
3,PTC,6,33.3%,-1.45%,4.96%,-8.25%,1.3d,6,16.7%,-2.67%,5.72%,-9.92%,1.5d,6,50.0%,-0.64%,5.72%,-11.27%,3.5d,6
4,ISRG,10,30.0%,-0.66%,7.40%,-7.27%,1.6d,10,30.0%,-1.29%,7.73%,-7.27%,2.2d,10,60.0%,0.54%,10.72%,-9.03%,4.3d,10
5,ETN,4,25.0%,-2.47%,4.41%,-10.84%,1.0d,4,50.0%,-1.75%,10.22%,-10.84%,2.8d,4,50.0%,-1.37%,10.22%,-13.75%,5.0d,4
6,DXCM,5,20.0%,0.46%,13.05%,-11.34%,1.0d,5,40.0%,-0.11%,13.70%,-12.26%,2.4d,5,40.0%,0.98%,20.17%,-12.26%,3.8d,5
7,NCLH,5,0.0%,-4.48%,5.96%,-9.14%,1.0d,5,40.0%,-1.54%,10.24%,-9.14%,1.4d,5,60.0%,1.62%,10.24%,-9.14%,4.6d,5


📊 4H Buy Signal Backtest Summary (2026-04-01)


,Ticker,Samples,3d Win Rate,3d Avg Return,3d Max Gain,3d Max Drawdown,3d Avg Peak Day,3d Valid Samples,5d Win Rate,5d Avg Return,5d Max Gain,5d Max Drawdown,5d Avg Peak Day,5d Valid Samples,8d Win Rate,8d Avg Return,8d Max Gain,8d Max Drawdown,8d Avg Peak Day,8d Valid Samples
0,ETN,10,70.0%,-0.12%,5.53%,-14.35%,2.1d,10,70.0%,1.17%,5.88%,-14.35%,3.5d,10,60.0%,0.25%,7.84%,-14.35%,5.3d,10
1,SWKS,16,50.0%,-0.15%,10.33%,-21.70%,1.6d,16,56.2%,0.58%,11.90%,-24.64%,2.6d,16,43.8%,0.29%,13.00%,-24.64%,3.8d,16
2,KMB,25,48.0%,-0.60%,4.64%,-16.73%,1.8d,25,56.0%,-1.23%,4.64%,-17.09%,2.9d,25,48.0%,-1.34%,4.64%,-17.41%,4.0d,25
3,MPWR,9,44.4%,1.27%,12.81%,-7.02%,1.4d,9,44.4%,0.80%,14.31%,-7.74%,2.6d,9,22.2%,-0.25%,20.59%,-17.79%,3.3d,9
4,DXCM,19,42.1%,0.62%,17.60%,-11.36%,1.5d,19,63.2%,2.61%,20.60%,-12.67%,3.1d,19,73.7%,1.69%,20.60%,-15.29%,4.5d,19
5,NCLH,15,40.0%,-0.92%,22.44%,-19.14%,1.1d,15,40.0%,0.41%,22.44%,-19.14%,2.3d,15,66.7%,2.35%,25.11%,-19.14%,4.8d,15
6,PTC,8,37.5%,-1.36%,4.44%,-13.30%,1.0d,8,37.5%,-0.75%,4.84%,-13.80%,2.1d,8,37.5%,-0.17%,5.39%,-13.80%,4.0d,8
7,ISRG,17,35.3%,-0.09%,17.60%,-8.31%,1.4d,17,29.4%,-0.01%,17.60%,-8.31%,2.1d,17,41.2%,0.53%,17.60%,-9.03%,3.6d,17



📄 Exported to 'scan_results' folder
